# Train DeepAR For Selected Clusters

This notebook installs dependencies, loads the repo, and calls the reusable training script in `scripts/train_deepar_clusters.py`.


In [2]:
!pip uninstall -y torch torchvision torchaudio lightning pytorch-lightning gluonts sympy
!pip install -q sympy==1.13.1
!pip install -q torch==2.5.1 torchvision==0.20.1 torchaudio==2.5.1 --index-url https://download.pytorch.org/whl/cu121
!pip install -q gluonts pyarrow holidays lightning==2.5.0.post0

Found existing installation: torch 2.10.0+cu128
Uninstalling torch-2.10.0+cu128:
  Successfully uninstalled torch-2.10.0+cu128
Found existing installation: torchvision 0.25.0+cu128
Uninstalling torchvision-0.25.0+cu128:
  Successfully uninstalled torchvision-0.25.0+cu128
Found existing installation: torchaudio 2.10.0+cu128
Uninstalling torchaudio-2.10.0+cu128:
  Successfully uninstalled torchaudio-2.10.0+cu128
Found existing installation: sympy 1.14.0
Uninstalling sympy-1.14.0:
  Successfully uninstalled sympy-1.14.0
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.2/6.2 MB 119.8 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 780.4/780.4 MB 1.3 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.3/7.3 MB 29.3 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.4/3.4 MB 125.7 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.7/23.7 MB 113.3 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 823.6/823.6 kB 66.9 MB/s eta 0:00:

In [3]:
import torch, torchvision, torchaudio, sympy
print("torch:", torch.__version__)
print("torchvision:", torchvision.__version__)
print("torchaudio:", torchaudio.__version__)
print("sympy:", sympy.__version__)
print("cuda:", torch.cuda.is_available())

torch: 2.5.1+cu121
torchvision: 0.20.1+cu121
torchaudio: 2.5.1+cu121
sympy: 1.13.1
cuda: True


In [26]:
import pandas as pd
import json
from train_deepar_clusters import TrainConfig, RandomSearchConfig, run_experiment, run_random_search
from gluonts.dataset.common import ListDataset
from train_deepar_clusters import (
    load_cluster_panels,
    build_metadata,
    build_dynamic_features,
    fit_predictor,
    set_seed,
)
from tqdm.auto import tqdm

In [4]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
%cd /content/drive/MyDrive/Electricity-Project

/content/drive/MyDrive/Electricity-Project


In [6]:
import torch
print('CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))

CUDA available: True
GPU: NVIDIA A100-SXM4-40GB


In [7]:
OUTPUT_DIR = None

HOLIDAY_COUNTRY = 'PT'

In [8]:
# Hyperparameters Tuning
RUN_RANDOM_SEARCH = True
N_TRIALS = 8
SEARCH_OUTPUT_DIR = None

SEARCH_SPACE = {
    'context_length_choices': (24 * 7, 24 * 14, 24 * 21),
    'hidden_size_choices': (48, 64, 96, 128),
    'dropout_rate_choices': (0.05, 0.1, 0.2, 0.3),
    'learning_rate_choices': (1e-3, 5e-4, 2e-4),
    'epochs_choices': (5, 10),
    'num_batches_per_epoch_choices': (50, 100),
    'batch_size_choices': (64,),
    'num_parallel_samples_choices': (50,),
}


# Cluster 2, 3

In [30]:
CLUSTER_IDS = (2, 3)

In [10]:
base_config = TrainConfig(
    train_path='train_hourly_preprocessed.parquet',
    test_path='test_hourly_preprocessed.parquet',
    cluster_path='clusters_3models.parquet',
    output_dir=OUTPUT_DIR,
    cluster_ids=CLUSTER_IDS,
    prediction_length=24,
    context_length=24 * 14,
    epochs=20,
    batch_size=64,
    num_batches_per_epoch=100,
    hidden_size=64,
    dropout_rate=0.1,
    use_hour_of_day=True,
    use_day_of_week=True,
    use_weekend=True,
    use_month=True,
    use_holiday=True,
    holiday_country=HOLIDAY_COUNTRY,
)

if RUN_RANDOM_SEARCH:
    search_config = RandomSearchConfig(
        n_trials=N_TRIALS,
        output_dir=SEARCH_OUTPUT_DIR,
        **SEARCH_SPACE,
    )
    results = run_random_search(base_config, search_config)
else:
    results = run_experiment(base_config)

results

[Trial 1/8] context=168, hidden=96, dropout=0.2, lr=0.001, epochs=10, batches=50


INFO: GPU available: True (cuda), used: True
INFO:lightning.pytorch.utilities.rank_zero:GPU available: True (cuda), used: True
INFO: TPU available: False, using: 0 TPU cores
INFO:lightning.pytorch.utilities.rank_zero:TPU available: False, using: 0 TPU cores
INFO: HPU available: False, using: 0 HPUs
INFO:lightning.pytorch.utilities.rank_zero:HPU available: False, using: 0 HPUs
/usr/local/lib/python3.12/dist-packages/lightning/pytorch/trainer/configuration_validator.py:70: You defined a `validation_step` but have no `val_dataloader`. Skipping val loop.
INFO: You are using a CUDA device ('NVIDIA A100-SXM4-40GB') that has Tensor Cores. To properly utilize them, you should set `torch.set_float32_matmul_precision('medium' | 'high')` which will trade-off precision for performance. For more details, read https://pytorch.org/docs/stable/generated/torch.set_float32_matmul_precision.html#torch.set_float32_matmul_precision
INFO:lightning.pytorch.utilities.rank_zero:You are using a CUDA device ('NV

Training: |          | 0/? [00:00<?, ?it/s]

INFO: Epoch 0, global step 50: 'train_loss' reached 7.15480 (best 7.15480), saving model to '/content/drive/MyDrive/Electricity-Project/checkpoints/epoch=0-step=50.ckpt' as top 1
INFO:lightning.pytorch.utilities.rank_zero:Epoch 0, global step 50: 'train_loss' reached 7.15480 (best 7.15480), saving model to '/content/drive/MyDrive/Electricity-Project/checkpoints/epoch=0-step=50.ckpt' as top 1
INFO: Epoch 1, global step 100: 'train_loss' reached 6.28267 (best 6.28267), saving model to '/content/drive/MyDrive/Electricity-Project/checkpoints/epoch=1-step=100.ckpt' as top 1
INFO:lightning.pytorch.utilities.rank_zero:Epoch 1, global step 100: 'train_loss' reached 6.28267 (best 6.28267), saving model to '/content/drive/MyDrive/Electricity-Project/checkpoints/epoch=1-step=100.ckpt' as top 1
INFO: Epoch 2, global step 150: 'train_loss' reached 6.07393 (best 6.07393), saving model to '/content/drive/MyDrive/Electricity-Project/checkpoints/epoch=2-step=150.ckpt' as top 1
INFO:lightning.pytorch.ut

Validation rolling forecast:   0%|          | 0/17756 [00:00<?, ?window/s]

...

INFO: GPU available: True (cuda), used: True
INFO:lightning.pytorch.utilities.rank_zero:GPU available: True (cuda), used: True
INFO: TPU available: False, using: 0 TPU cores
INFO:lightning.pytorch.utilities.rank_zero:TPU available: False, using: 0 TPU cores
INFO: HPU available: False, using: 0 HPUs
INFO:lightning.pytorch.utilities.rank_zero:HPU available: False, using: 0 HPUs
/usr/local/lib/python3.12/dist-packages/lightning/pytorch/trainer/configuration_validator.py:70: You defined a `validation_step` but have no `val_dataloader`. Skipping val loop.
/usr/local/lib/python3.12/dist-packages/lightning/pytorch/callbacks/model_checkpoint.py:654: Checkpoint directory /content/drive/MyDrive/Electricity-Project/checkpoints exists and is not empty.
INFO: LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]
INFO:lightning.pytorch.accelerators.cuda:LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]
INFO: 
  | Name  | Type        | Params | Mode  | In sizes                                                      | Out 

Training: |          | 0/? [00:00<?, ?it/s]

INFO: Epoch 0, global step 100: 'train_loss' reached 7.20432 (best 7.20432), saving model to '/content/drive/MyDrive/Electricity-Project/checkpoints/epoch=0-step=100.ckpt' as top 1
INFO:lightning.pytorch.utilities.rank_zero:Epoch 0, global step 100: 'train_loss' reached 7.20432 (best 7.20432), saving model to '/content/drive/MyDrive/Electricity-Project/checkpoints/epoch=0-step=100.ckpt' as top 1
INFO: Epoch 1, global step 200: 'train_loss' reached 6.18444 (best 6.18444), saving model to '/content/drive/MyDrive/Electricity-Project/checkpoints/epoch=1-step=200.ckpt' as top 1
INFO:lightning.pytorch.utilities.rank_zero:Epoch 1, global step 200: 'train_loss' reached 6.18444 (best 6.18444), saving model to '/content/drive/MyDrive/Electricity-Project/checkpoints/epoch=1-step=200.ckpt' as top 1
INFO: Epoch 2, global step 300: 'train_loss' reached 5.91778 (best 5.91778), saving model to '/content/drive/MyDrive/Electricity-Project/checkpoints/epoch=2-step=300.ckpt' as top 1
INFO:lightning.pytorc

Validation rolling forecast:   0%|          | 0/17756 [00:00<?, ?window/s]

{'summary_df':    trial  overall_mape  overall_wmape  overall_epsilon_mape  context_length  \
 0      7      7.263247       6.213088              7.683969             504   
 1      3      7.158662       6.307977              7.553069             336   
 2      2      8.394817       6.747770              8.808058             168   
 3      8      7.998184       6.939348              8.409334             168   
 4      5      7.888569       7.156149              8.270409             168   
 5      4      9.166676       7.286303              9.598702             504   
 6      1      8.319133       7.609014              8.699319             168   
 7      6      8.822342       7.855966              9.219142             168   
 
    hidden_size  dropout_rate  learning_rate  epochs  num_batches_per_epoch  \
 0          128          0.05         0.0005      10                     50   
 1          128          0.10         0.0010      10                    100   
 2           48          0.

In [11]:
summary_path = "outputs/deepar_random_search_2_3/random_search_summary.csv"
results_path = "outputs/deepar_random_search_2_3/random_search_results.json"

summary_df = pd.read_csv(summary_path)
summary_df.head()

,trial,overall_mape,overall_wmape,overall_epsilon_mape,context_length,hidden_size,dropout_rate,learning_rate,epochs,num_batches_per_epoch,batch_size,num_parallel_samples,prediction_path,cluster_2_mape,cluster_3_mape,cluster_2_wmape,cluster_3_wmape,cluster_2_epsilon_mape,cluster_3_epsilon_mape
0,7,7.263247,6.213088,7.683969,504,128,0.05,0.0005,10,50,64,50,outputs/deepar_random_search_2_3/trial_07/deep...,6.118415,13.246160,5.684308,8.486128,6.403359,14.376193
1,3,7.158662,6.307977,7.553069,336,128,0.10,0.0010,10,100,64,50,outputs/deepar_random_search_2_3/trial_03/deep...,6.193130,12.204550,5.814528,8.429145,6.463395,13.247492
2,2,8.394817,6.747770,8.808058,168,48,0.10,0.0010,5,100,64,50,outputs/deepar_random_search_2_3/trial_02/deep...,7.215106,14.560013,6.273687,8.785689,7.500079,15.643301
3,8,7.998184,6.939348,8.409334,168,48,0.05,0.0005,5,100,64,50,outputs/deepar_random_search_2_3/trial_08/deep...,6.937115,13.543353,6.505316,8.805100,7.233807,14.552414
4,5,7.888569,7.156149,8.270409,168,128,0.30,0.0010,5,100,64,50,outputs/deepar_random_search_2_3/trial_05/deep...,7.140871,11.796054,6.937071,8.097890,7.406096,12.787138


In [12]:
best_row = summary_df.iloc[0]
best_row

,0
trial,7
overall_mape,7.263247
overall_wmape,6.213088
overall_epsilon_mape,7.683969
context_length,504
hidden_size,128
dropout_rate,0.05
learning_rate,0.0005
epochs,10
num_batches_per_epoch,50


In [13]:
best_cols = [
    "overall_wmape", "overall_mape",
    "context_length", "hidden_size", "dropout_rate", "learning_rate",
    "epochs", "num_batches_per_epoch", "batch_size", "num_parallel_samples"
]

extra_cols = [
    c for c in summary_df.columns
    if c.startswith("cluster_") and c.endswith(("wmape", "mape"))
]

summary_df.loc[[0], best_cols + extra_cols].T

,0
overall_wmape,6.213088
overall_mape,7.263247
context_length,504.000000
hidden_size,128.000000
dropout_rate,0.050000
learning_rate,0.000500
epochs,10.000000
num_batches_per_epoch,50.000000
batch_size,64.000000
num_parallel_samples,50.000000


In [14]:
best_pred_path = best_row["prediction_path"]
best_pred_df = pd.read_parquet(best_pred_path)
best_pred_df.head()

,timestamp,meter_id,cluster_id,y_true,y_pred,forecast_start,ape,epsilon_ape
0,2014-07-01 00:00:00,MT_098,3,1219.110379,1367.979004,2014-07-01,12.211251,12.201242
1,2014-07-01 01:00:00,MT_098,3,784.184514,963.058228,2014-07-01,22.810156,22.781106
2,2014-07-01 02:00:00,MT_098,3,747.940692,880.813232,2014-07-01,17.765117,17.741397
3,2014-07-01 03:00:00,MT_098,3,777.594728,830.005798,2014-07-01,6.740152,6.731496
4,2014-07-01 04:00:00,MT_098,3,780.889621,751.439392,2014-07-01,3.771369,3.766546


In [31]:
best_config = TrainConfig(
    train_path="train_hourly_preprocessed.parquet",
    test_path="test_hourly_preprocessed.parquet",
    cluster_path="clusters_3models.parquet",
    cluster_ids=CLUSTER_IDS,
    output_dir=OUTPUT_DIR,
    freq="h",
    prediction_length=24,
    context_length=int(best_row["context_length"]),
    epochs=int(best_row["epochs"]),
    learning_rate=float(best_row["learning_rate"]),
    batch_size=int(best_row["batch_size"]),
    num_batches_per_epoch=int(best_row["num_batches_per_epoch"]),
    num_layers=2,
    hidden_size=int(best_row["hidden_size"]),
    dropout_rate=float(best_row["dropout_rate"]),
    num_parallel_samples=int(best_row["num_parallel_samples"]),
    seed=42,
    use_hour_of_day=True,
    use_day_of_week=True,
    use_weekend=True,
    use_month=True,
    use_holiday=True,
    holiday_country=HOLIDAY_COUNTRY,
)

In [32]:
set_seed(best_config.seed)

train_panel, test_panel, labels = load_cluster_panels(best_config)

history_panel = pd.concat([train_panel, test_panel]).sort_index()

meta = build_metadata(history_panel, labels, best_config)

future_start = history_panel.index.max() + pd.Timedelta(hours=1)
future_end = (future_start.to_period("M") + 2).end_time.floor("h")
future_index = pd.date_range(future_start, future_end, freq="h")

print("history:", history_panel.index.min(), "->", history_panel.index.max(), history_panel.shape)
print("future :", future_index.min(), "->", future_index.max(), len(future_index))

dynamic_feat_df = build_dynamic_features(history_panel.index, future_index, best_config)

predictor = fit_predictor(history_panel, meta, dynamic_feat_df.loc[history_panel.index.union(future_index)], best_config)

print("final predictor trained")


history: 2012-10-01 00:00:00 -> 2014-12-31 23:00:00 (19728, 193)
future : 2015-01-01 00:00:00 -> 2015-03-31 23:00:00 2160


INFO: GPU available: True (cuda), used: True
INFO:lightning.pytorch.utilities.rank_zero:GPU available: True (cuda), used: True
INFO: TPU available: False, using: 0 TPU cores
INFO:lightning.pytorch.utilities.rank_zero:TPU available: False, using: 0 TPU cores
INFO: HPU available: False, using: 0 HPUs
INFO:lightning.pytorch.utilities.rank_zero:HPU available: False, using: 0 HPUs
/usr/local/lib/python3.12/dist-packages/lightning/pytorch/trainer/configuration_validator.py:70: You defined a `validation_step` but have no `val_dataloader`. Skipping val loop.
/usr/local/lib/python3.12/dist-packages/lightning/pytorch/callbacks/model_checkpoint.py:654: Checkpoint directory /content/drive/MyDrive/Electricity-Project/checkpoints exists and is not empty.
INFO: LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]
INFO:lightning.pytorch.accelerators.cuda:LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]
INFO: 
  | Name  | Type        | Params | Mode  | In sizes                                                         | O

Training: |          | 0/? [00:00<?, ?it/s]

INFO: Epoch 0, global step 50: 'train_loss' reached 7.23476 (best 7.23476), saving model to '/content/drive/MyDrive/Electricity-Project/checkpoints/epoch=0-step=50.ckpt' as top 1
INFO:lightning.pytorch.utilities.rank_zero:Epoch 0, global step 50: 'train_loss' reached 7.23476 (best 7.23476), saving model to '/content/drive/MyDrive/Electricity-Project/checkpoints/epoch=0-step=50.ckpt' as top 1
INFO: Epoch 1, global step 100: 'train_loss' reached 6.16488 (best 6.16488), saving model to '/content/drive/MyDrive/Electricity-Project/checkpoints/epoch=1-step=100.ckpt' as top 1
INFO:lightning.pytorch.utilities.rank_zero:Epoch 1, global step 100: 'train_loss' reached 6.16488 (best 6.16488), saving model to '/content/drive/MyDrive/Electricity-Project/checkpoints/epoch=1-step=100.ckpt' as top 1
INFO: Epoch 2, global step 150: 'train_loss' was not in top 1
INFO:lightning.pytorch.utilities.rank_zero:Epoch 2, global step 150: 'train_loss' was not in top 1
INFO: Epoch 3, global step 200: 'train_loss' 

final predictor trained


In [33]:
future_pred_frames = []
step = best_config.prediction_length  # 24

total_windows = sum((len(future_index) + step - 1) // step for _ in history_panel.columns)
progress = tqdm(total=total_windows, desc="Future rolling forecast", unit="window")

for meter_id in history_panel.columns:
    history = history_panel[meter_id].astype(float).copy()

    static_cat = [int(meta.loc[meter_id, "cluster_code"])]
    static_real = [float(meta.loc[meter_id, "log_mean_hourly_load"])]
    cluster_id = int(meta.loc[meter_id, "cluster_id"])

    meter_future_frames = []

    for start_idx in range(0, len(future_index), step):
        end_idx = min(start_idx + step, len(future_index))
        horizon = end_idx - start_idx
        forecast_slice = future_index[start_idx:end_idx]

        record = {
            "item_id": str(meter_id),
            "start": history.index[0],
            "target": history.to_numpy(),
            "feat_static_cat": static_cat,
            "feat_static_real": static_real,
        }

        if not dynamic_feat_df.empty:
            dynamic_index = history.index.append(forecast_slice)
            record["feat_dynamic_real"] = dynamic_feat_df.loc[dynamic_index].to_numpy(dtype=float).T

        ds = ListDataset([record], freq=best_config.freq)
        forecast = next(predictor.predict(ds, num_samples=best_config.num_parallel_samples))

        pred_mean = np.asarray(forecast.mean[:horizon], dtype=float)

        meter_future_frames.append(
            pd.DataFrame(
                {
                    "timestamp": forecast_slice,
                    "meter_id": meter_id,
                    "cluster_id": cluster_id,
                    "y_pred": pred_mean,
                }
            )
        )

        history_extension = pd.Series(pred_mean, index=forecast_slice)
        history = pd.concat([history, history_extension])

        progress.update(1)

    future_pred_frames.append(pd.concat(meter_future_frames, ignore_index=True))

progress.close()

future_pred_df = pd.concat(future_pred_frames, ignore_index=True)
future_pred_df.head()


Future rolling forecast:   0%|          | 0/17370 [00:00<?, ?window/s]

,timestamp,meter_id,cluster_id,y_pred
0,2015-01-01 00:00:00,MT_098,3,983.492920
1,2015-01-01 01:00:00,MT_098,3,753.809631
2,2015-01-01 02:00:00,MT_098,3,778.887573
3,2015-01-01 03:00:00,MT_098,3,748.515991
4,2015-01-01 04:00:00,MT_098,3,702.720642


In [34]:
cluster_future_df = (
    future_pred_df.groupby(["timestamp", "cluster_id"], as_index=False)["y_pred"]
    .sum()
    .sort_values(["cluster_id", "timestamp"])
)

cluster_future_df.head()


,timestamp,cluster_id,y_pred
0,2015-01-01 00:00:00,2,207886.626015
2,2015-01-01 01:00:00,2,192406.921200
4,2015-01-01 02:00:00,2,200499.144157
6,2015-01-01 03:00:00,2,205478.383068
8,2015-01-01 04:00:00,2,216566.189610


In [35]:
future_pred_path = "outputs/future_3months_predictions_2_3.parquet"
cluster_future_path = "outputs/future_3months_cluster_sum_2_3.parquet"

future_pred_df.to_parquet(future_pred_path, index=False)
cluster_future_df.to_parquet(cluster_future_path, index=False)

print("saved meter-level forecast to:", future_pred_path)
print("saved cluster-level forecast to:", cluster_future_path)


saved meter-level forecast to: outputs/future_3months_predictions_2_3.parquet
saved cluster-level forecast to: outputs/future_3months_cluster_sum_2_3.parquet


# Cluster 1, 11

In [15]:
CLUSTER_IDS = (1, 11)

In [16]:
base_config_2 = TrainConfig(
    train_path='train_hourly_preprocessed.parquet',
    test_path='test_hourly_preprocessed.parquet',
    cluster_path='clusters_3models.parquet',
    output_dir=OUTPUT_DIR,
    cluster_ids=CLUSTER_IDS,
    prediction_length=24,
    context_length=24 * 14,
    epochs=20,
    batch_size=64,
    num_batches_per_epoch=100,
    hidden_size=64,
    dropout_rate=0.1,
    use_hour_of_day=True,
    use_day_of_week=True,
    use_weekend=True,
    use_month=True,
    use_holiday=True,
    holiday_country=HOLIDAY_COUNTRY,
)

if RUN_RANDOM_SEARCH:
    search_config_2 = RandomSearchConfig(
        n_trials=N_TRIALS,
        output_dir=SEARCH_OUTPUT_DIR,
        **SEARCH_SPACE,
    )
    results_2 = run_random_search(base_config_2, search_config_2)
else:
    results_2 = run_experiment(base_config_2)

results_2

[Trial 1/8] context=168, hidden=96, dropout=0.2, lr=0.001, epochs=10, batches=50


INFO: GPU available: True (cuda), used: True
INFO:lightning.pytorch.utilities.rank_zero:GPU available: True (cuda), used: True
INFO: TPU available: False, using: 0 TPU cores
INFO:lightning.pytorch.utilities.rank_zero:TPU available: False, using: 0 TPU cores
INFO: HPU available: False, using: 0 HPUs
INFO:lightning.pytorch.utilities.rank_zero:HPU available: False, using: 0 HPUs
/usr/local/lib/python3.12/dist-packages/lightning/pytorch/trainer/configuration_validator.py:70: You defined a `validation_step` but have no `val_dataloader`. Skipping val loop.
/usr/local/lib/python3.12/dist-packages/lightning/pytorch/callbacks/model_checkpoint.py:654: Checkpoint directory /content/drive/MyDrive/Electricity-Project/checkpoints exists and is not empty.
INFO: LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]
INFO:lightning.pytorch.accelerators.cuda:LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]
INFO: 
  | Name  | Type        | Params | Mode  | In sizes                                                      | Out 

Training: |          | 0/? [00:00<?, ?it/s]

INFO: Epoch 0, global step 50: 'train_loss' reached 5.81283 (best 5.81283), saving model to '/content/drive/MyDrive/Electricity-Project/checkpoints/epoch=0-step=50.ckpt' as top 1
INFO:lightning.pytorch.utilities.rank_zero:Epoch 0, global step 50: 'train_loss' reached 5.81283 (best 5.81283), saving model to '/content/drive/MyDrive/Electricity-Project/checkpoints/epoch=0-step=50.ckpt' as top 1
INFO: Epoch 1, global step 100: 'train_loss' reached 5.14165 (best 5.14165), saving model to '/content/drive/MyDrive/Electricity-Project/checkpoints/epoch=1-step=100.ckpt' as top 1
INFO:lightning.pytorch.utilities.rank_zero:Epoch 1, global step 100: 'train_loss' reached 5.14165 (best 5.14165), saving model to '/content/drive/MyDrive/Electricity-Project/checkpoints/epoch=1-step=100.ckpt' as top 1
INFO: Epoch 2, global step 150: 'train_loss' reached 4.96525 (best 4.96525), saving model to '/content/drive/MyDrive/Electricity-Project/checkpoints/epoch=2-step=150.ckpt' as top 1
INFO:lightning.pytorch.ut

Validation rolling forecast:   0%|          | 0/7360 [00:00<?, ?window/s]

...

INFO: GPU available: True (cuda), used: True
INFO:lightning.pytorch.utilities.rank_zero:GPU available: True (cuda), used: True
INFO: TPU available: False, using: 0 TPU cores
INFO:lightning.pytorch.utilities.rank_zero:TPU available: False, using: 0 TPU cores
INFO: HPU available: False, using: 0 HPUs
INFO:lightning.pytorch.utilities.rank_zero:HPU available: False, using: 0 HPUs
/usr/local/lib/python3.12/dist-packages/lightning/pytorch/trainer/configuration_validator.py:70: You defined a `validation_step` but have no `val_dataloader`. Skipping val loop.
/usr/local/lib/python3.12/dist-packages/lightning/pytorch/callbacks/model_checkpoint.py:654: Checkpoint directory /content/drive/MyDrive/Electricity-Project/checkpoints exists and is not empty.
INFO: LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]
INFO:lightning.pytorch.accelerators.cuda:LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]
INFO: 
  | Name  | Type        | Params | Mode  | In sizes                                                      | Out 

Training: |          | 0/? [00:00<?, ?it/s]

INFO: Epoch 0, global step 100: 'train_loss' reached 5.89476 (best 5.89476), saving model to '/content/drive/MyDrive/Electricity-Project/checkpoints/epoch=0-step=100.ckpt' as top 1
INFO:lightning.pytorch.utilities.rank_zero:Epoch 0, global step 100: 'train_loss' reached 5.89476 (best 5.89476), saving model to '/content/drive/MyDrive/Electricity-Project/checkpoints/epoch=0-step=100.ckpt' as top 1
INFO: Epoch 1, global step 200: 'train_loss' reached 5.02179 (best 5.02179), saving model to '/content/drive/MyDrive/Electricity-Project/checkpoints/epoch=1-step=200.ckpt' as top 1
INFO:lightning.pytorch.utilities.rank_zero:Epoch 1, global step 200: 'train_loss' reached 5.02179 (best 5.02179), saving model to '/content/drive/MyDrive/Electricity-Project/checkpoints/epoch=1-step=200.ckpt' as top 1
INFO: Epoch 2, global step 300: 'train_loss' reached 4.86128 (best 4.86128), saving model to '/content/drive/MyDrive/Electricity-Project/checkpoints/epoch=2-step=300.ckpt' as top 1
INFO:lightning.pytorc

Validation rolling forecast:   0%|          | 0/7360 [00:00<?, ?window/s]

{'summary_df':    trial  overall_mape  overall_wmape  overall_epsilon_mape  context_length  \
 0      7      9.301961       8.945734              9.299005             504   
 1      8      9.735734       9.151205              9.746721             168   
 2      3      9.685706       9.230726              9.674900             336   
 3      1      9.698676       9.276410              9.688791             168   
 4      4      9.813473       9.456947              9.818808             504   
 5      2      9.866521       9.527440              9.860616             168   
 6      6      9.859788       9.593709              9.861438             168   
 7      5     11.133687      10.341550             11.115143             168   
 
    hidden_size  dropout_rate  learning_rate  epochs  num_batches_per_epoch  \
 0          128          0.05         0.0005      10                     50   
 1           48          0.05         0.0005       5                    100   
 2          128          0.

In [8]:
summary_path = "outputs/deepar_random_search_1_11/random_search_summary.csv"
results_path = "outputs/deepar_random_search_1_11/random_search_results.json"

summary_df = pd.read_csv(summary_path)
summary_df.head()

,trial,overall_mape,overall_wmape,overall_epsilon_mape,context_length,hidden_size,dropout_rate,learning_rate,epochs,num_batches_per_epoch,batch_size,num_parallel_samples,prediction_path,cluster_1_mape,cluster_11_mape,cluster_1_wmape,cluster_11_wmape,cluster_1_epsilon_mape,cluster_11_epsilon_mape
0,7,9.301961,8.945734,9.299005,504,128,0.05,0.0005,10,50,64,50,outputs/deepar_random_search_1_11/trial_07/dee...,9.405956,8.007439,9.180733,5.968048,9.385229,8.235566
1,8,9.735734,9.151205,9.746721,168,48,0.05,0.0005,5,100,64,50,outputs/deepar_random_search_1_11/trial_08/dee...,9.888031,7.839970,9.439020,5.504273,9.866732,8.266587
2,3,9.685706,9.230726,9.674900,336,128,0.10,0.0010,10,100,64,50,outputs/deepar_random_search_1_11/trial_03/dee...,9.809545,8.144169,9.479271,6.081382,9.785018,8.316781
3,1,9.698676,9.276410,9.688791,168,96,0.20,0.0010,10,50,64,50,outputs/deepar_random_search_1_11/trial_01/dee...,9.841957,7.915131,9.560899,5.671630,9.820551,8.063748
4,4,9.813473,9.456947,9.818808,504,128,0.05,0.0005,5,50,64,50,outputs/deepar_random_search_1_11/trial_04/dee...,9.958859,8.003723,9.749827,5.745834,9.941856,8.301217


In [9]:
best_row = summary_df.iloc[0]
best_row

,0
trial,7
overall_mape,9.301961
overall_wmape,8.945734
overall_epsilon_mape,9.299005
context_length,504
hidden_size,128
dropout_rate,0.05
learning_rate,0.0005
epochs,10
num_batches_per_epoch,50


In [10]:
best_cols = [
    "overall_wmape", "overall_mape",
    "context_length", "hidden_size", "dropout_rate", "learning_rate",
    "epochs", "num_batches_per_epoch", "batch_size", "num_parallel_samples"
]

extra_cols = [
    c for c in summary_df.columns
    if c.startswith("cluster_") and c.endswith(("wmape", "mape"))
]

summary_df.loc[[0], best_cols + extra_cols].T

,0
overall_wmape,8.945734
overall_mape,9.301961
context_length,504.000000
hidden_size,128.000000
dropout_rate,0.050000
learning_rate,0.000500
epochs,10.000000
num_batches_per_epoch,50.000000
batch_size,64.000000
num_parallel_samples,50.000000


In [11]:
best_pred_path = best_row["prediction_path"]
best_pred_df = pd.read_parquet(best_pred_path)
best_pred_df.head()

,timestamp,meter_id,cluster_id,y_true,y_pred,forecast_start,ape,epsilon_ape
0,2014-07-01 00:00:00,MT_003,1,6.950478,7.248984,2014-07-01,4.294762,3.754573
1,2014-07-01 01:00:00,MT_003,1,6.950478,6.874405,2014-07-01,1.094493,0.956829
2,2014-07-01 02:00:00,MT_003,1,6.950478,6.828506,2014-07-01,1.754863,1.534139
3,2014-07-01 03:00:00,MT_003,1,6.950478,7.155840,2014-07-01,2.954654,2.583021
4,2014-07-01 04:00:00,MT_003,1,6.950478,6.601655,2014-07-01,5.018695,4.387451


In [23]:
best_config = TrainConfig(
    train_path="train_hourly_preprocessed.parquet",
    test_path="test_hourly_preprocessed.parquet",
    cluster_path="clusters_3models.parquet",
    cluster_ids=CLUSTER_IDS,
    output_dir=OUTPUT_DIR,
    freq="h",
    prediction_length=24,
    context_length=int(best_row["context_length"]),
    epochs=int(best_row["epochs"]),
    learning_rate=float(best_row["learning_rate"]),
    batch_size=int(best_row["batch_size"]),
    num_batches_per_epoch=int(best_row["num_batches_per_epoch"]),
    num_layers=2,
    hidden_size=int(best_row["hidden_size"]),
    dropout_rate=float(best_row["dropout_rate"]),
    num_parallel_samples=int(best_row["num_parallel_samples"]),
    seed=42,
    use_hour_of_day=True,
    use_day_of_week=True,
    use_weekend=True,
    use_month=True,
    use_holiday=True,
    holiday_country=HOLIDAY_COUNTRY,
)


In [24]:
set_seed(best_config.seed)

train_panel, test_panel, labels = load_cluster_panels(best_config)

history_panel = pd.concat([train_panel, test_panel]).sort_index()

meta = build_metadata(history_panel, labels, best_config)

future_start = history_panel.index.max() + pd.Timedelta(hours=1)
future_end = (future_start.to_period("M") + 2).end_time.floor("h")
future_index = pd.date_range(future_start, future_end, freq="h")

print("history:", history_panel.index.min(), "->", history_panel.index.max(), history_panel.shape)
print("future :", future_index.min(), "->", future_index.max(), len(future_index))

dynamic_feat_df = build_dynamic_features(history_panel.index, future_index, best_config)

predictor = fit_predictor(history_panel, meta, dynamic_feat_df.loc[history_panel.index.union(future_index)], best_config)

print("final predictor trained")


history: 2012-10-01 00:00:00 -> 2014-12-31 23:00:00 (19728, 80)
future : 2015-01-01 00:00:00 -> 2015-03-31 23:00:00 2160


INFO: GPU available: True (cuda), used: True
INFO:lightning.pytorch.utilities.rank_zero:GPU available: True (cuda), used: True
INFO: TPU available: False, using: 0 TPU cores
INFO:lightning.pytorch.utilities.rank_zero:TPU available: False, using: 0 TPU cores
INFO: HPU available: False, using: 0 HPUs
INFO:lightning.pytorch.utilities.rank_zero:HPU available: False, using: 0 HPUs
/usr/local/lib/python3.12/dist-packages/lightning/pytorch/trainer/configuration_validator.py:70: You defined a `validation_step` but have no `val_dataloader`. Skipping val loop.
/usr/local/lib/python3.12/dist-packages/lightning/pytorch/callbacks/model_checkpoint.py:654: Checkpoint directory /content/drive/MyDrive/Electricity-Project/checkpoints exists and is not empty.
INFO: LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]
INFO:lightning.pytorch.accelerators.cuda:LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]
INFO: 
  | Name  | Type        | Params | Mode  | In sizes                                                         | O

Training: |          | 0/? [00:00<?, ?it/s]

INFO: Epoch 0, global step 50: 'train_loss' reached 5.94466 (best 5.94466), saving model to '/content/drive/MyDrive/Electricity-Project/checkpoints/epoch=0-step=50.ckpt' as top 1
INFO:lightning.pytorch.utilities.rank_zero:Epoch 0, global step 50: 'train_loss' reached 5.94466 (best 5.94466), saving model to '/content/drive/MyDrive/Electricity-Project/checkpoints/epoch=0-step=50.ckpt' as top 1
INFO: Epoch 1, global step 100: 'train_loss' reached 5.06495 (best 5.06495), saving model to '/content/drive/MyDrive/Electricity-Project/checkpoints/epoch=1-step=100.ckpt' as top 1
INFO:lightning.pytorch.utilities.rank_zero:Epoch 1, global step 100: 'train_loss' reached 5.06495 (best 5.06495), saving model to '/content/drive/MyDrive/Electricity-Project/checkpoints/epoch=1-step=100.ckpt' as top 1
INFO: Epoch 2, global step 150: 'train_loss' reached 4.82090 (best 4.82090), saving model to '/content/drive/MyDrive/Electricity-Project/checkpoints/epoch=2-step=150.ckpt' as top 1
INFO:lightning.pytorch.ut

final predictor trained


In [27]:
future_pred_frames = []
step = best_config.prediction_length  # 24

total_windows = sum((len(future_index) + step - 1) // step for _ in history_panel.columns)
progress = tqdm(total=total_windows, desc="Future rolling forecast", unit="window")

for meter_id in history_panel.columns:
    history = history_panel[meter_id].astype(float).copy()

    static_cat = [int(meta.loc[meter_id, "cluster_code"])]
    static_real = [float(meta.loc[meter_id, "log_mean_hourly_load"])]
    cluster_id = int(meta.loc[meter_id, "cluster_id"])

    meter_future_frames = []

    for start_idx in range(0, len(future_index), step):
        end_idx = min(start_idx + step, len(future_index))
        horizon = end_idx - start_idx
        forecast_slice = future_index[start_idx:end_idx]

        record = {
            "item_id": str(meter_id),
            "start": history.index[0],
            "target": history.to_numpy(),
            "feat_static_cat": static_cat,
            "feat_static_real": static_real,
        }

        if not dynamic_feat_df.empty:
            dynamic_index = history.index.append(forecast_slice)
            record["feat_dynamic_real"] = dynamic_feat_df.loc[dynamic_index].to_numpy(dtype=float).T

        ds = ListDataset([record], freq=best_config.freq)
        forecast = next(predictor.predict(ds, num_samples=best_config.num_parallel_samples))

        pred_mean = np.asarray(forecast.mean[:horizon], dtype=float)

        meter_future_frames.append(
            pd.DataFrame(
                {
                    "timestamp": forecast_slice,
                    "meter_id": meter_id,
                    "cluster_id": cluster_id,
                    "y_pred": pred_mean,
                }
            )
        )

        history_extension = pd.Series(pred_mean, index=forecast_slice)
        history = pd.concat([history, history_extension])

        progress.update(1)

    future_pred_frames.append(pd.concat(meter_future_frames, ignore_index=True))

progress.close()

future_pred_df = pd.concat(future_pred_frames, ignore_index=True)
future_pred_df.head()


Future rolling forecast:   0%|          | 0/7200 [00:00<?, ?window/s]

,timestamp,meter_id,cluster_id,y_pred
0,2015-01-01 00:00:00,MT_003,1,6.203574
1,2015-01-01 01:00:00,MT_003,1,5.253814
2,2015-01-01 02:00:00,MT_003,1,5.149548
3,2015-01-01 03:00:00,MT_003,1,5.341709
4,2015-01-01 04:00:00,MT_003,1,5.085263


In [28]:
cluster_future_df = (
    future_pred_df.groupby(["timestamp", "cluster_id"], as_index=False)["y_pred"]
    .sum()
    .sort_values(["cluster_id", "timestamp"])
)

cluster_future_df.head()


,timestamp,cluster_id,y_pred
0,2015-01-01 00:00:00,1,45621.253894
2,2015-01-01 01:00:00,1,39739.008689
4,2015-01-01 02:00:00,1,37294.647842
6,2015-01-01 03:00:00,1,37588.624073
8,2015-01-01 04:00:00,1,35235.923749


In [29]:
future_pred_path = "outputs/future_3months_predictions_1_11.parquet"
cluster_future_path = "outputs/future_3months_cluster_sum_1_11.parquet"

future_pred_df.to_parquet(future_pred_path, index=False)
cluster_future_df.to_parquet(cluster_future_path, index=False)

print("saved meter-level forecast to:", future_pred_path)
print("saved cluster-level forecast to:", cluster_future_path)


saved meter-level forecast to: outputs/future_3months_predictions_1_11.parquet
saved cluster-level forecast to: outputs/future_3months_cluster_sum_1_11.parquet


# Cluster 7

In [16]:
CLUSTER_IDS = (7,)

In [17]:
# Hyperparameters Tuning
RUN_RANDOM_SEARCH = True
N_TRIALS = 8
SEARCH_OUTPUT_DIR = None

SEARCH_SPACE = {
    'context_length_choices': (24 * 7, 24 * 14),
    'hidden_size_choices': (32, 48, 64),
    'dropout_rate_choices': (0.05, 0.1, 0.2),
    'learning_rate_choices': (1e-3, 5e-4),
    'epochs_choices': (5, 10),
    'num_batches_per_epoch_choices': (30, 50),
    'batch_size_choices': (32, 64),
    'num_parallel_samples_choices': (50,),
}

In [18]:
base_config_3 = TrainConfig(
    train_path='train_hourly_preprocessed.parquet',
    test_path='test_hourly_preprocessed.parquet',
    cluster_path='clusters_3models.parquet',
    output_dir=OUTPUT_DIR,
    cluster_ids=CLUSTER_IDS,
    prediction_length=24,
    context_length=24 * 14,
    epochs=20,
    batch_size=64,
    num_batches_per_epoch=100,
    hidden_size=64,
    dropout_rate=0.1,
    use_hour_of_day=True,
    use_day_of_week=True,
    use_weekend=True,
    use_month=True,
    use_holiday=True,
    holiday_country=HOLIDAY_COUNTRY,
)

if RUN_RANDOM_SEARCH:
    search_config_3 = RandomSearchConfig(
        n_trials=N_TRIALS,
        output_dir=SEARCH_OUTPUT_DIR,
        **SEARCH_SPACE,
    )
    results_3 = run_random_search(base_config_3, search_config_3)
else:
    results_3 = run_experiment(base_config_3)

results_3

[Trial 1/8] context=168, hidden=64, dropout=0.1, lr=0.001, epochs=10, batches=30


INFO: GPU available: True (cuda), used: True
INFO:lightning.pytorch.utilities.rank_zero:GPU available: True (cuda), used: True
INFO: TPU available: False, using: 0 TPU cores
INFO:lightning.pytorch.utilities.rank_zero:TPU available: False, using: 0 TPU cores
INFO: HPU available: False, using: 0 HPUs
INFO:lightning.pytorch.utilities.rank_zero:HPU available: False, using: 0 HPUs
/usr/local/lib/python3.12/dist-packages/lightning/pytorch/trainer/configuration_validator.py:70: You defined a `validation_step` but have no `val_dataloader`. Skipping val loop.
INFO: You are using a CUDA device ('NVIDIA A100-SXM4-40GB') that has Tensor Cores. To properly utilize them, you should set `torch.set_float32_matmul_precision('medium' | 'high')` which will trade-off precision for performance. For more details, read https://pytorch.org/docs/stable/generated/torch.set_float32_matmul_precision.html#torch.set_float32_matmul_precision
INFO:lightning.pytorch.utilities.rank_zero:You are using a CUDA device ('NV

Training: |          | 0/? [00:00<?, ?it/s]

INFO: Epoch 0, global step 30: 'train_loss' reached 7.19592 (best 7.19592), saving model to '/content/drive/MyDrive/Electricity-Project/checkpoints/epoch=0-step=30.ckpt' as top 1
INFO:lightning.pytorch.utilities.rank_zero:Epoch 0, global step 30: 'train_loss' reached 7.19592 (best 7.19592), saving model to '/content/drive/MyDrive/Electricity-Project/checkpoints/epoch=0-step=30.ckpt' as top 1
INFO: Epoch 1, global step 60: 'train_loss' reached 6.25648 (best 6.25648), saving model to '/content/drive/MyDrive/Electricity-Project/checkpoints/epoch=1-step=60.ckpt' as top 1
INFO:lightning.pytorch.utilities.rank_zero:Epoch 1, global step 60: 'train_loss' reached 6.25648 (best 6.25648), saving model to '/content/drive/MyDrive/Electricity-Project/checkpoints/epoch=1-step=60.ckpt' as top 1
INFO: Epoch 2, global step 90: 'train_loss' reached 6.08888 (best 6.08888), saving model to '/content/drive/MyDrive/Electricity-Project/checkpoints/epoch=2-step=90.ckpt' as top 1
INFO:lightning.pytorch.utilitie

Validation rolling forecast:   0%|          | 0/1656 [00:00<?, ?window/s]

...

INFO: GPU available: True (cuda), used: True
INFO:lightning.pytorch.utilities.rank_zero:GPU available: True (cuda), used: True
INFO: TPU available: False, using: 0 TPU cores
INFO:lightning.pytorch.utilities.rank_zero:TPU available: False, using: 0 TPU cores
INFO: HPU available: False, using: 0 HPUs
INFO:lightning.pytorch.utilities.rank_zero:HPU available: False, using: 0 HPUs
/usr/local/lib/python3.12/dist-packages/lightning/pytorch/trainer/configuration_validator.py:70: You defined a `validation_step` but have no `val_dataloader`. Skipping val loop.
/usr/local/lib/python3.12/dist-packages/lightning/pytorch/callbacks/model_checkpoint.py:654: Checkpoint directory /content/drive/MyDrive/Electricity-Project/checkpoints exists and is not empty.
INFO: LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]
INFO:lightning.pytorch.accelerators.cuda:LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]
INFO: 
  | Name  | Type        | Params | Mode  | In sizes                                                      | Out 

Training: |          | 0/? [00:00<?, ?it/s]

INFO: Epoch 0, global step 30: 'train_loss' reached 7.23406 (best 7.23406), saving model to '/content/drive/MyDrive/Electricity-Project/checkpoints/epoch=0-step=30.ckpt' as top 1
INFO:lightning.pytorch.utilities.rank_zero:Epoch 0, global step 30: 'train_loss' reached 7.23406 (best 7.23406), saving model to '/content/drive/MyDrive/Electricity-Project/checkpoints/epoch=0-step=30.ckpt' as top 1
INFO: Epoch 1, global step 60: 'train_loss' reached 6.19536 (best 6.19536), saving model to '/content/drive/MyDrive/Electricity-Project/checkpoints/epoch=1-step=60.ckpt' as top 1
INFO:lightning.pytorch.utilities.rank_zero:Epoch 1, global step 60: 'train_loss' reached 6.19536 (best 6.19536), saving model to '/content/drive/MyDrive/Electricity-Project/checkpoints/epoch=1-step=60.ckpt' as top 1
INFO: Epoch 2, global step 90: 'train_loss' reached 6.14735 (best 6.14735), saving model to '/content/drive/MyDrive/Electricity-Project/checkpoints/epoch=2-step=90.ckpt' as top 1
INFO:lightning.pytorch.utilitie

Validation rolling forecast:   0%|          | 0/1656 [00:00<?, ?window/s]

{'summary_df':    trial  overall_mape  overall_wmape  overall_epsilon_mape  context_length  \
 0      1      5.703177       4.889144             55.669890             168   
 1      4      5.676155       4.977045             52.239740             336   
 2      3      5.874045       5.010893             55.058443             168   
 3      7      5.894007       5.254286             52.773744             168   
 4      2      6.695483       5.800654             56.658923             168   
 5      6      6.745140       5.963651             56.108672             168   
 6      5      6.739250       5.988199             51.796970             168   
 7      8     11.866295      10.381764             71.197272             168   
 
    hidden_size  dropout_rate  learning_rate  epochs  num_batches_per_epoch  \
 0           64          0.10         0.0010      10                     30   
 1           32          0.05         0.0010      10                     50   
 2           32          0.

In [19]:
summary_path = "outputs/deepar_random_search_7/random_search_summary.csv"
results_path = "outputs/deepar_random_search_7/random_search_results.json"

summary_df = pd.read_csv(summary_path)
summary_df.head()

,trial,overall_mape,overall_wmape,overall_epsilon_mape,context_length,hidden_size,dropout_rate,learning_rate,epochs,num_batches_per_epoch,batch_size,num_parallel_samples,prediction_path,cluster_7_mape,cluster_7_wmape,cluster_7_epsilon_mape
0,1,5.703177,4.889144,55.669890,168,64,0.10,0.0010,10,30,32,50,outputs/deepar_random_search_7/trial_01/deepar...,5.703177,4.889144,55.669890
1,4,5.676155,4.977045,52.239740,336,32,0.05,0.0010,10,50,32,50,outputs/deepar_random_search_7/trial_04/deepar...,5.676155,4.977045,52.239740
2,3,5.874045,5.010893,55.058443,168,32,0.20,0.0005,10,50,32,50,outputs/deepar_random_search_7/trial_03/deepar...,5.874045,5.010893,55.058443
3,7,5.894007,5.254286,52.773744,168,32,0.05,0.0010,5,50,32,50,outputs/deepar_random_search_7/trial_07/deepar...,5.894007,5.254286,52.773744
4,2,6.695483,5.800654,56.658923,168,48,0.05,0.0010,10,30,64,50,outputs/deepar_random_search_7/trial_02/deepar...,6.695483,5.800654,56.658923


In [20]:
best_row = summary_df.iloc[0]
best_row

,0
trial,1
overall_mape,5.703177
overall_wmape,4.889144
overall_epsilon_mape,55.66989
context_length,168
hidden_size,64
dropout_rate,0.1
learning_rate,0.001
epochs,10
num_batches_per_epoch,30


In [21]:
best_cols = [
    "overall_wmape", "overall_mape",
    "context_length", "hidden_size", "dropout_rate", "learning_rate",
    "epochs", "num_batches_per_epoch", "batch_size", "num_parallel_samples"
]

extra_cols = [
    c for c in summary_df.columns
    if c.startswith("cluster_") and c.endswith(("wmape", "mape"))
]

summary_df.loc[[0], best_cols + extra_cols].T

,0
overall_wmape,4.889144
overall_mape,5.703177
context_length,168.000000
hidden_size,64.000000
dropout_rate,0.100000
learning_rate,0.001000
epochs,10.000000
num_batches_per_epoch,30.000000
batch_size,32.000000
num_parallel_samples,50.000000


In [22]:
best_pred_path = best_row["prediction_path"]
best_pred_df = pd.read_parquet(best_pred_path)
best_pred_df.head()

,timestamp,meter_id,cluster_id,y_true,y_pred,forecast_start,ape,epsilon_ape
0,2014-07-01 00:00:00,MT_002,7,106.685633,100.081825,2014-07-01,6.189969,6.132487
1,2014-07-01 01:00:00,MT_002,7,88.193457,88.362907,2014-07-01,0.192135,0.189981
2,2014-07-01 02:00:00,MT_002,7,84.637269,84.708855,2014-07-01,0.084580,0.083592
3,2014-07-01 03:00:00,MT_002,7,83.926031,81.498848,2014-07-01,2.892051,2.857997
4,2014-07-01 04:00:00,MT_002,7,89.615932,84.852402,2014-07-01,5.315495,5.256835


In [23]:
best_config = TrainConfig(
    train_path="train_hourly_preprocessed.parquet",
    test_path="test_hourly_preprocessed.parquet",
    cluster_path="clusters_3models.parquet",
    cluster_ids=CLUSTER_IDS,
    output_dir=OUTPUT_DIR,
    freq="h",
    prediction_length=24,
    context_length=int(best_row["context_length"]),
    epochs=int(best_row["epochs"]),
    learning_rate=float(best_row["learning_rate"]),
    batch_size=int(best_row["batch_size"]),
    num_batches_per_epoch=int(best_row["num_batches_per_epoch"]),
    num_layers=2,
    hidden_size=int(best_row["hidden_size"]),
    dropout_rate=float(best_row["dropout_rate"]),
    num_parallel_samples=int(best_row["num_parallel_samples"]),
    seed=42,
    use_hour_of_day=True,
    use_day_of_week=True,
    use_weekend=True,
    use_month=True,
    use_holiday=True,
    holiday_country=HOLIDAY_COUNTRY,
)


In [24]:
set_seed(best_config.seed)

train_panel, test_panel, labels = load_cluster_panels(best_config)

history_panel = pd.concat([train_panel, test_panel]).sort_index()

meta = build_metadata(history_panel, labels, best_config)

future_start = history_panel.index.max() + pd.Timedelta(hours=1)
future_end = (future_start.to_period("M") + 2).end_time.floor("h")
future_index = pd.date_range(future_start, future_end, freq="h")

print("history:", history_panel.index.min(), "->", history_panel.index.max(), history_panel.shape)
print("future :", future_index.min(), "->", future_index.max(), len(future_index))

dynamic_feat_df = build_dynamic_features(history_panel.index, future_index, best_config)

predictor = fit_predictor(history_panel, meta, dynamic_feat_df.loc[history_panel.index.union(future_index)], best_config)

print("final predictor trained")


INFO: GPU available: True (cuda), used: True
INFO:lightning.pytorch.utilities.rank_zero:GPU available: True (cuda), used: True
INFO: TPU available: False, using: 0 TPU cores
INFO:lightning.pytorch.utilities.rank_zero:TPU available: False, using: 0 TPU cores
INFO: HPU available: False, using: 0 HPUs
INFO:lightning.pytorch.utilities.rank_zero:HPU available: False, using: 0 HPUs
/usr/local/lib/python3.12/dist-packages/lightning/pytorch/trainer/configuration_validator.py:70: You defined a `validation_step` but have no `val_dataloader`. Skipping val loop.
/usr/local/lib/python3.12/dist-packages/lightning/pytorch/callbacks/model_checkpoint.py:654: Checkpoint directory /content/drive/MyDrive/Electricity-Project/checkpoints exists and is not empty.
INFO: LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]
INFO:lightning.pytorch.accelerators.cuda:LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]


history: 2012-10-01 00:00:00 -> 2014-12-31 23:00:00 (19728, 18)
future : 2015-01-01 00:00:00 -> 2015-03-31 23:00:00 2160


INFO: 
  | Name  | Type        | Params | Mode  | In sizes                                                      | Out sizes   
-----------------------------------------------------------------------------------------------------------------------------
0 | model | DeepARModel | 62.9 K | train | [[1, 1], [1, 1], [1, 888, 6], [1, 888], [1, 888], [1, 24, 6]] | [1, 100, 24]
-----------------------------------------------------------------------------------------------------------------------------
62.9 K    Trainable params
0         Non-trainable params
62.9 K    Total params
0.252     Total estimated model params size (MB)
11        Modules in train mode
0         Modules in eval mode
INFO:lightning.pytorch.callbacks.model_summary:
  | Name  | Type        | Params | Mode  | In sizes                                                      | Out sizes   
-----------------------------------------------------------------------------------------------------------------------------
0 | model | De

Training: |          | 0/? [00:00<?, ?it/s]

INFO: Epoch 0, global step 30: 'train_loss' reached 7.19417 (best 7.19417), saving model to '/content/drive/MyDrive/Electricity-Project/checkpoints/epoch=0-step=30.ckpt' as top 1
INFO:lightning.pytorch.utilities.rank_zero:Epoch 0, global step 30: 'train_loss' reached 7.19417 (best 7.19417), saving model to '/content/drive/MyDrive/Electricity-Project/checkpoints/epoch=0-step=30.ckpt' as top 1
INFO: Epoch 1, global step 60: 'train_loss' reached 6.24994 (best 6.24994), saving model to '/content/drive/MyDrive/Electricity-Project/checkpoints/epoch=1-step=60.ckpt' as top 1
INFO:lightning.pytorch.utilities.rank_zero:Epoch 1, global step 60: 'train_loss' reached 6.24994 (best 6.24994), saving model to '/content/drive/MyDrive/Electricity-Project/checkpoints/epoch=1-step=60.ckpt' as top 1
INFO: Epoch 2, global step 90: 'train_loss' reached 5.99041 (best 5.99041), saving model to '/content/drive/MyDrive/Electricity-Project/checkpoints/epoch=2-step=90.ckpt' as top 1
INFO:lightning.pytorch.utilitie

final predictor trained


In [27]:
future_pred_frames = []
step = best_config.prediction_length  # 24

total_windows = sum((len(future_index) + step - 1) // step for _ in history_panel.columns)
progress = tqdm(total=total_windows, desc="Future rolling forecast", unit="window")

for meter_id in history_panel.columns:
    history = history_panel[meter_id].astype(float).copy()

    static_cat = [int(meta.loc[meter_id, "cluster_code"])]
    static_real = [float(meta.loc[meter_id, "log_mean_hourly_load"])]
    cluster_id = int(meta.loc[meter_id, "cluster_id"])

    meter_future_frames = []

    for start_idx in range(0, len(future_index), step):
        end_idx = min(start_idx + step, len(future_index))
        horizon = end_idx - start_idx
        forecast_slice = future_index[start_idx:end_idx]

        record = {
            "item_id": str(meter_id),
            "start": history.index[0],
            "target": history.to_numpy(),
            "feat_static_cat": static_cat,
            "feat_static_real": static_real,
        }

        if not dynamic_feat_df.empty:
            dynamic_index = history.index.append(forecast_slice)
            record["feat_dynamic_real"] = dynamic_feat_df.loc[dynamic_index].to_numpy(dtype=float).T

        ds = ListDataset([record], freq=best_config.freq)
        forecast = next(predictor.predict(ds, num_samples=best_config.num_parallel_samples))

        pred_mean = np.asarray(forecast.mean[:horizon], dtype=float)

        meter_future_frames.append(
            pd.DataFrame(
                {
                    "timestamp": forecast_slice,
                    "meter_id": meter_id,
                    "cluster_id": cluster_id,
                    "y_pred": pred_mean,
                }
            )
        )

        history_extension = pd.Series(pred_mean, index=forecast_slice)
        history = pd.concat([history, history_extension])

        progress.update(1)

    future_pred_frames.append(pd.concat(meter_future_frames, ignore_index=True))

progress.close()

future_pred_df = pd.concat(future_pred_frames, ignore_index=True)
future_pred_df.head()


Future rolling forecast:   0%|          | 0/1620 [00:00<?, ?window/s]

,timestamp,meter_id,cluster_id,y_pred
0,2015-01-01 00:00:00,MT_002,7,80.311684
1,2015-01-01 01:00:00,MT_002,7,76.433128
2,2015-01-01 02:00:00,MT_002,7,75.637680
3,2015-01-01 03:00:00,MT_002,7,72.254333
4,2015-01-01 04:00:00,MT_002,7,73.069855


In [28]:
cluster_future_df = (
    future_pred_df.groupby(["timestamp", "cluster_id"], as_index=False)["y_pred"]
    .sum()
    .sort_values(["cluster_id", "timestamp"])
)

cluster_future_df.head()


,timestamp,cluster_id,y_pred
0,2015-01-01 00:00:00,7,40978.793724
1,2015-01-01 01:00:00,7,41221.941338
2,2015-01-01 02:00:00,7,40732.187233
3,2015-01-01 03:00:00,7,39581.415077
4,2015-01-01 04:00:00,7,38815.508118


In [29]:
future_pred_path = "outputs/future_3months_predictions_7.parquet"
cluster_future_path = "outputs/future_3months_cluster_sum_7.parquet"

future_pred_df.to_parquet(future_pred_path, index=False)
cluster_future_df.to_parquet(cluster_future_path, index=False)

print("saved meter-level forecast to:", future_pred_path)
print("saved cluster-level forecast to:", cluster_future_path)


saved meter-level forecast to: outputs/future_3months_predictions_7.parquet
saved cluster-level forecast to: outputs/future_3months_cluster_sum_7.parquet
